# Multiscale Hybrid diffusion with Robin local problems

Barrenechea, Gomes and Paredes (2024), [DOI 10.1137/22M1542556](https://doi.org/10.1137/22M1542556). The local operator is coercive Robin, and the global multiplier is $\lambda=(q-p\sigma)\cdot n$, with $q=-K\nabla p$ and $\sigma=\nu(x-a)/2$. It is not the physical normal flux.

This notebook runs a small analytical patch and reads the separate multilevel campaign. The polygonal and triangular mesh connectivity is explicitly chosen for this package; the campaign is not a digitized reproduction of the article figures.

In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.mesh import FaceSpace, SkeletonSpace, TriangleMesh
from pymhm.mh import solve_mh

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pixi.toml").exists())
RESULTS = ROOT / "examples/results/mh"
FIGURES = ROOT / "docs/figures/mh"

## Small nonzero Dirichlet patch

For $K=I$, $p=1+x^2+y^2$, the source is $f=-4$ and the physical flux is $q=(-2x,-2y)$. Local P2 and trace P2 represent the Robin multiplier exactly. Testing the local equation with one checks $\int_{\partial K}(\lambda+p\sigma\cdot n)=\int_K f$.

In [ ]:
mesh = TriangleMesh.unit_square(2)
skeleton = SkeletonSpace(mesh, tuple(FaceSpace.uniform(2) for _ in mesh.faces))
exact = lambda x: 1 + np.sum(x**2, axis=1)
with threadpool_limits(1):
    solution = solve_mh(mesh, source=-4, dirichlet=exact, skeleton=skeleton, degree=2, local_refinement=2)
pressure_error = solution.l2_error(exact)
flux_error = solution.flux_l2_error(lambda x: -2*x)
balance = float(np.max(abs(solution.conservation_residuals())))
assert max(pressure_error, flux_error, balance) < 1e-10
{"pressure_L2": pressure_error, "physical_flux_L2": flux_error, "macro_balance_max": balance}

## Five-resolution analytical campaign

The published equation has $p=\sin(6\pi x)\sin(14\pi y)$ and $f=232\pi^2p$. Both degree pairs $(\ell,k)=(1,3),(2,4)$ are measured on triangles and on nonconvex L-shaped polygons. The recorded fields use broken, independent interface values. The campaign explicitly retains extended-precision corrections at unchanged residual thresholds; it runs separately from this notebook.

In [ ]:
record = json.loads((RESULTS / "comparison.json").read_text())
assert not record["source_changed_during_run"]
[(row["mesh"], row["trace_degree"], row["resolution"], row["pressure_relative_error"], row["flux_relative_error"]) for row in record["smooth"]]

In [ ]:
display(Image(filename=str(FIGURES / "convergence.png")))
display(Image(filename=str(FIGURES / "triangles-fields.png")))
display(Image(filename=str(FIGURES / "L-polygons-fields.png")))

## The same-space limit as $\nu\to0$

The pressure difference is integrated in the broken energy norm with identical fine meshes, trace spaces, data and quadrature. The reference is the MHM solution in those spaces, not an exact solution. Local and global spectral conditioning is reported separately; no runtime speedup follows from these condition numbers.

In [ ]:
record["vanishing_robin"]["rows"]

In [ ]:
display(Image(filename=str(FIGURES / "vanishing-robin.png")))

To acquire all numerical levels again:

```bash
pixi run -e notebooks python -m examples.mh_campaign --workers 8
pixi run -e notebooks python -m examples.plot_mh
```

The current API prescribes pressure on every exterior face. It does not accept physical Neumann conditions. A certified lower permeability eigenvalue is required for material callbacks; arbitrary trace/local spaces do not inherit a uniform stability result from a successful linear solve.